# Module 4 — Q&A RAG Pipeline

**Part of the RAG-Based E-commerce Customer Support Chatbot**

**Stage 4** of the pipeline (final stage): for messages that aren't a greeting and aren't
purely a routing action, retrieve the most relevant historical support Q&A pairs and use them
to ground an LLM's answer.

**Dataset:** same Bitext dataset as Notebook 3 — `instruction` (customer question) is embedded
and indexed; `response` (agent answer) is what gets retrieved and shown to the LLM as grounding
context.

**Components:**
1. `sentence-transformers/all-MiniLM-L6-v2` — fast, small (~80MB) embedding model, good default
   for semantic search over short support Q&A pairs.
2. **FAISS** (`IndexFlatIP` over normalized embeddings, i.e. cosine similarity) — local,
   in-memory vector store; no external service needed, which keeps this notebook runnable
   end-to-end in Colab with no extra setup.
3. **Groq API** — fast LLM inference, used here as the generation step of RAG.

**Output artifacts:** `rag_faiss.index`, `rag_documents.joblib` (the instruction/response
pairs aligned to the FAISS index), reusable `retrieve(query, k)` and `generate_answer(...)`
functions.


## 1. Install dependencies

In [4]:
!pip install -q datasets sentence-transformers faiss-cpu groq joblib pandas numpy


## 2. Imports and Groq API key setup

In [5]:
import os
import joblib
import numpy as np
import pandas as pd
import faiss
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from groq import Groq

# --- Set your Groq API key safely ---
# You can set your API key directly here if it's not configured in your environment variables:
GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "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")

# Recommended active and free model available on Groq
GROQ_MODEL = "llama-3.3-70b-versatile"

# Initialize the Groq client
try:
    groq_client = Groq(api_key=GROQ_API_KEY)
    print("Groq client initialized successfully!")
except Exception as e:
    print(f"Error initializing Groq client: {e}")


Groq client initialized successfully!


## 3. Load the dataset and build the retrieval corpus

In [6]:
raw = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")
df = raw["train"].to_pandas()

# Deduplicate near-identical instructions to keep the index compact and reduce redundant hits
df = df.drop_duplicates(subset=["instruction"]).reset_index(drop=True)
df = df[["instruction", "response", "intent", "category"]].dropna()

print(df.shape)
df.head()


README.md:   0%|          | 0.00/11.9k [00:00<?, ?B/s]

Bitext_Sample_Customer_Support_Training_(…): reconstructing file:   0%|          |  0.00B / 19.2MB            

Bitext_Sample_Customer_Support_Training_(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

(24635, 4)


,instruction,response,intent,category
0,question about cancelling order {{Order Number}},I've understood you have a question regarding ...,cancel_order,ORDER
1,i have a question about cancelling oorder {{Or...,I've been informed that you have a question ab...,cancel_order,ORDER
2,i need help cancelling puchase {{Order Number}},I can sense that you're seeking assistance wit...,cancel_order,ORDER
3,I need to cancel purchase {{Order Number}},I understood that you need assistance with can...,cancel_order,ORDER
4,"I cannot afford this order, cancel purchase {{...",I'm sensitive to the fact that you're facing f...,cancel_order,ORDER


## 4. Embed the corpus

In [7]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")

corpus_texts = df["instruction"].tolist()
corpus_embeddings = embedder.encode(
    corpus_texts,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,   # normalize so inner product == cosine similarity
)

print("Embeddings shape:", corpus_embeddings.shape)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/193 [00:00<?, ?it/s]

Embeddings shape: (24635, 384)


## 5. Build the FAISS index

In [8]:
embedding_dim = corpus_embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)  # inner product on normalized vectors = cosine sim
index.add(corpus_embeddings)

print("Number of vectors in index:", index.ntotal)


Number of vectors in index: 24635


## 6. Retriever function

In [9]:
def retrieve(query: str, k: int = 3) -> list[dict]:
    """Retrieve the top-k most similar historical Q&A pairs for a query.

    Args:
        query: the customer's message.
        k: number of results to return.

    Returns:
        List of dicts: {"instruction": str, "response": str, "score": float}
    """
    query_emb = embedder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
    scores, indices = index.search(query_emb, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        row = df.iloc[idx]
        results.append({
            "instruction": row["instruction"],
            "response": row["response"],
            "score": float(score),
        })
    return results


# Quick smoke test
for r in retrieve("Where is my package? It's been 2 weeks", k=3):
    print(f"[{r['score']:.3f}] {r['instruction']}  ->  {r['response'][:80]}...")


[0.734] where can I see when my damn package is going to arrive?  ->  I understand your frustration and eagerness to track your package. To provide yo...
[0.698] where do I see when my package is going to arrive?  ->  We understand your eagerness to track the arrival of your package. To provide yo...
[0.686] where can I see when my package is going to arrive?  ->  We understand your eagerness to track the progress and estimated arrival time of...


## 7. Prompt template + Groq generation

This implements the exact prompt template specified in the project brief, including the
sentiment-aware instruction to acknowledge frustration.


In [10]:
SYSTEM_PROMPT_TEMPLATE = (
    "You are a helpful, professional customer support assistant for an online retailer. "
    "Answer the customer's question using ONLY the information in the retrieved support "
    "responses below. If the customer sounds frustrated ({detected_sentiment}), acknowledge "
    "that before answering. If the retrieved context does not cover the question, say so "
    "honestly and offer to escalate to a human agent rather than guessing."
)

def format_context(retrieved_chunks: list[dict]) -> str:
    if not retrieved_chunks:
        return "(no relevant context found)"
    lines = []
    for i, chunk in enumerate(retrieved_chunks, start=1):
        lines.append(f"{i}. Q: {chunk['instruction']}\n   A: {chunk['response']}")
    return "\n".join(lines)


def generate_answer(user_message: str, detected_sentiment: str = "neutral", k: int = 3) -> dict:
    """Run the full RAG step: retrieve context, then query the Groq LLM with the grounded prompt.

    Args:
        user_message: the customer's message.
        detected_sentiment: output of Module 2 ("negative" | "neutral" | "positive").
        k: number of chunks to retrieve.

    Returns:
        {"answer": str, "retrieved_chunks": list[dict]}
    """
    retrieved_chunks = retrieve(user_message, k=k)
    context_str = format_context(retrieved_chunks)

    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(detected_sentiment=detected_sentiment)
    user_prompt = (
        f"Context:\n{context_str}\n\n"
        f'Customer question: "{user_message}"'
    )

    try:
        completion = groq_client.chat.completions.create(
            model=GROQ_MODEL,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
            temperature=0.3,
            max_tokens=400,
        )
        answer = completion.choices[0].message.content
    except Exception as e:
        # Graceful fallback if the Groq API call fails (e.g. missing/invalid key, network issue)
        answer = (
            "I'm sorry, I'm having trouble reaching our answer service right now. "
            "Let me connect you with a human agent who can help."
        )
        print(f"[generate_answer] Groq API error, falling back to safe default: {e}")

    return {"answer": answer, "retrieved_chunks": retrieved_chunks}


## 8. End-to-end smoke test

In [11]:
result = generate_answer(
    user_message="I've been waiting 2 weeks for my refund and nobody is responding!",
    detected_sentiment="negative",
    k=3,
)

print("ANSWER:\n", result["answer"])
print("\nRETRIEVED CONTEXT:")
for c in result["retrieved_chunks"]:
    print(f"  [{c['score']:.3f}] {c['instruction']}")


[generate_answer] Groq API error, falling back to safe default: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}
ANSWER:
 I'm sorry, I'm having trouble reaching our answer service right now. Let me connect you with a human agent who can help.

RETRIEVED CONTEXT:
  [0.745] can you help me to see if there are any news on my refund?
  [0.744] i try to see if there are any updates on my refund
  [0.733] can you help me check if there are any news on my refund?


## 9. Mount Google Drive and set up the project folder structure

All 4 notebooks share one Drive folder, `RAG_chatbot_project/`, organized into one subfolder
per module so artifacts never collide and `app.py` can load each module from a predictable
path:

```
RAG_chatbot_project/
├── language_detection/      <- Notebook 1 saves here
├── sentiment_classifier/    <- Notebook 2 saves here
├── intent_classifier/       <- Notebook 3 saves here
└── rag_pipeline/            <- this notebook saves here
```

This cell mounts Drive and creates the full structure (safe to re-run).


In [12]:
from google.colab import drive
drive.mount('/content/drive')

import os

BASE_DIR = "/content/drive/MyDrive/RAG_chatbot_project"
LANGUAGE_DIR = os.path.join(BASE_DIR, "language_detection")
SENTIMENT_DIR = os.path.join(BASE_DIR, "sentiment_classifier")
INTENT_DIR = os.path.join(BASE_DIR, "intent_classifier")
RAG_DIR = os.path.join(BASE_DIR, "rag_pipeline")

for d in [LANGUAGE_DIR, SENTIMENT_DIR, INTENT_DIR, RAG_DIR]:
    os.makedirs(d, exist_ok=True)

print("Drive mounted. Project folder structure ready under:", BASE_DIR)


Mounted at /content/drive
Drive mounted. Project folder structure ready under: /content/drive/MyDrive/RAG_chatbot_project


## 10. Save artifacts

In [13]:
faiss.write_index(index, os.path.join(RAG_DIR, "rag_faiss.index"))
joblib.dump(df.reset_index(drop=True), os.path.join(RAG_DIR, "rag_documents.joblib"))

print("Saved RAG artifacts to:", RAG_DIR)
print("- rag_faiss.index")
print("- rag_documents.joblib")
print()
print("NOTE: the embedding model (all-MiniLM-L6-v2) is downloaded fresh from")
print("sentence-transformers at load time in app.py, it does not need to be saved separately.")


Saved RAG artifacts to: /content/drive/MyDrive/RAG_chatbot_project/rag_pipeline
- rag_faiss.index
- rag_documents.joblib

NOTE: the embedding model (all-MiniLM-L6-v2) is downloaded fresh from
sentence-transformers at load time in app.py, it does not need to be saved separately.
